In [1]:
import json
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "amici2014response")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Amici_2014_Primates_REHY_exp1.csv")
complete_path_2 = os.path.join(original_data_pathway, "Amici_2014_Primates_REHY_exp2.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

experiment_import = [[df1, "Task_1_Imitation_1_human", "1"],
                        [df2, "Task_2_Imitation_1b_ape", "2"]]
for x, y, k in experiment_import:
    x['experiment_name']=y
    x['experiment']=k
df2['trial'] = df2['Session'] 



In [3]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x=x.rename(columns={"subjects": "ape"})
    x['study_id']="amici2014response"
    x['data_subset']="data_subset_" + str(index+1)
    x['ape'].replace('', np.nan, inplace=True)
    x.dropna(subset=['ape'], inplace=True)
    x = x.rename(columns={"day": "day_original"})
    x = x.rename(columns={"species": "species_original"})
    x['day_original']= pd.to_datetime(x['day_original'],format='%m/%d/%Y')
    x['year']= x['day_original'].dt.year
    x['month']= x['day_original'].dt.month
    x['day']= x['day_original'].dt.day
    data_frames[index]=x
new_df=data_frames[0]


In [4]:

fulldf = pd.concat(data_frames, ignore_index=True, sort=False)
# fulldf.columns

In [5]:

fulldf['ape'] = fulldf['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')
# fulldf.columns

In [6]:
fulldf.rename(columns={"ape": "participant"}, inplace=True)
fulldf['condition'].replace(' ', '_', inplace=True, regex=True)

In [ ]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

In [7]:
fulldf=fulldf[['study_id', 'experiment','experiment_name', 'year', 'month', 'day',  
        'participant', 'age_in_years','sex','species', 'session', 'trial',  'condition', 'yawn_baseline',
       'yawn_experimental', 'nosewipe_baseline', 'nosewipe_experimental',
       'scratch_baseline', 'scratch_experimental', 'wristshake_baseline', 'wristshake_experimental',
       'handclose_baseline', 'handclose_experimental', 'yawn_pre', 'yawn_video', 'yawn_post', 'nosewipe_pre',
       'nosewipe_video', 'nosewipe_post', 'scratch_pre', 'scratch_video', 'scratch_post' ]]

In [8]:
for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'amici2014response_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'amici2014response_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)